<a href="https://colab.research.google.com/github/Gan4x4/cv/blob/main/Prod/onnxruntime/1_onnxrt_colab.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Goal

We want to use [onnxruntime](https://onnxruntime.ai/docs/) with the [TensorRT](https://onnxruntime.ai/docs/execution-providers/TensorRT-ExecutionProvider.html) or [TensorRT-RTX](https://onnxruntime.ai/docs/execution-providers/TensorRTRTX-ExecutionProvider.html) provider for maximum inference speed.


# Obtain a Model in ONNX Format

First, we need a model in ONNX format.
Let's create one by converting [ResNet18](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.resnet18.html) using the [onnx](https://pypi.org/project/onnx/) and [onnxscript](https://pypi.org/project/onnxscript/) packages.

If you already have a converted model, you can skip this step.


In [1]:
# Installing converter
!pip install onnx onnxscript


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 23.2 MB/s eta 0:00:00


Now we can convert the PyTorch model to **ONNX** format.


In [2]:
import torch
import torchvision.models as models

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1) # ResNet-18 pretrained on ImageNet
model.eval()
dummy = torch.randn(1, 3, 224, 224) # Dummy input: NCHW = (batch, channels, height, width)

torch.onnx.export(
    model,
    dummy,
    "model.onnx",
    input_names=["INPUT__0"], # NPUT__0 - arbitrary string
    output_names=["OUTPUT__0"],
    opset_version=18,
    dynamic_axes={ # For using batches of differen sizes
        "INPUT__0": {0: "BATCH"},
        "OUTPUT__0": {0: "BATCH"},
    },
)

print("Saved: model.onnx")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 181MB/s]
/tmp/ipykernel_1464/1986971654.py:8: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0406 12:55:40.104000 1464 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0406 12:55:40.105000 1464 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0406 12:55:40.107000 1464 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, outpu

[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 41 of general pattern rewrite rules.
Saved: model.onnx


After conversion, there should be two files on disk: `model.onnx` (model structure) and `model.onnx.data` (model weights).

# Run ONNX Runtime

To run this model efficiently, we need to install [onnxruntime](https://onnxruntime.ai/docs/) with a suitable provider. For NVIDIA GPUs, the most powerful provider is [TensorRT](https://onnxruntime.ai/docs/execution-providers/TensorRT-ExecutionProvider.html), so we will use it.


# Issue

1. ONNX Runtime has many versions, for example:

- onnxruntime
- onnxruntime-gpu
- onnxruntime-openvino
...

Some providers, like OpenVINO, have their own specific packages, whereas TensorRT does not. However, during inference, you should still use an import like this:

> import onnxruntime

regardless of which package you installed.

This means **you cannot install different ONNX Runtime packages in the same environment**, even if they have different names.


In [3]:
# Install default packages
!pip install onnxruntime-gpu tensorrt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.8 MB/s eta 0:00:00
  Created wheel for tensorrt: filename=tensorrt-10.16.0.72-py3-none-any.whl size=16566 sha256=ef98dacf38f4281d1fac903fb2d6b0a88790e3e5c43e15b0af253fc4101dbb50
  Stored in directory: /root/.cache/pip/wheels/94/5f/04/7f93fac79967299afd72fc4feedfa60dbd27352e553e94582b
  Created wheel for tensorrt_cu13: filename=tensorrt_cu13-10.16.0.72-py3-none-any.whl size=23077 sha256=ca5a0cef3b277c4b558d8d84155ed9305b25d7afdc1a9e6b64b322bd172be0fd
  S

# Run

Let's launch the converted model.


In [4]:
import numpy as np
import onnxruntime as ort


sess = ort.InferenceSession("model.onnx",
                            providers=["TensorrtExecutionProvider"])
dummy = {"INPUT__0": np.random.rand(1, 3, 224, 224).astype(np.float32)}
out = sess.run(None, dummy)

print(f"providers={sess.get_providers()}  output={out[0].shape}")

*************** EP Error ***************
EP Error /onnxruntime_src/onnxruntime/python/onnxruntime_pybind_state.cc:539 void onnxruntime::python::RegisterTensorRTPluginsAsCustomOps(PySessionOptions&, const onnxruntime::ProviderOptions&) Please install TensorRT libraries as mentioned in the GPU requirements page, make sure they're in the PATH or LD_LIBRARY_PATH, and that your GPU is supported.
 when using ['TensorrtExecutionProvider']
Falling back to ['CPUExecutionProvider'] and retrying.
****************************************
providers=['CPUExecutionProvider']  output=(1, 1000)


`TensorrtExecutionProvider` should be in the list.


## Check Paths

If something goes wrong, inspect the library paths first.


In [5]:
import site, subprocess
site_packages = site.getsitepackages()[0]
target = f"{site_packages}/onnxruntime/capi/libonnxruntime_providers_tensorrt.so"
ldd_out = subprocess.check_output(["ldd", target], text=True)

groups = {"TensorRT": ["libnvinfer.so", "libnvonnxparser.so"], "CUDA": ["libcudart.so"], "cuDNN": ["libcudnn.so"]}

for label, prefixes in groups.items():
    print(f"{label}:")
    for prefix in prefixes:
        line = next((l for l in ldd_out.splitlines() if prefix in l), None)
        name = line.split("=>")[0].strip() if line and "=>" in line else prefix
        path = line.split("=>")[1].split("(")[0].strip() if line and "=>" in line else "not found"
        print(f"\t{name} | {path}")

TensorRT:
	libnvinfer.so.10 | not found
	libnvonnxparser.so.10 | not found
CUDA:
	libcudart.so.12 | /usr/local/cuda/targets/x86_64-linux/lib/libcudart.so.12
cuDNN:
	libcudnn.so.9 | /lib/x86_64-linux-gnu/libcudnn.so.9


Where are the TensorRT libraries installed?


In [6]:
from pathlib import Path
import site

site_packages = Path(site.getsitepackages()[0])
wanted_prefixes = ["libnvinfer.so", "libnvonnxparser.so"]

for prefix in wanted_prefixes:
    # Find all matching files and take the first one found, or 'not found'
    match = next(site_packages.rglob(f"{prefix}*"), "not found")
    print(f"{prefix} | {match}")

libnvinfer.so | /usr/local/lib/python3.12/dist-packages/tensorrt_libs/libnvinfer.so.10
libnvonnxparser.so | /usr/local/lib/python3.12/dist-packages/tensorrt_libs/libnvonnxparser.so.10


# Fix

In Colab or other container-based projects, the best fix is to install TensorRT system-wide rather than with `pip`.

But if you need to work inside a Python virtual environment, update `LD_LIBRARY_PATH` to include the TensorRT paths (`libnvinfer.so`, `libnvonnxparser.so`).


In [7]:
!apt-get install tensorrt

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libnvinfer-bin libnvinfer-dev libnvinfer-dispatch-dev libnvinfer-dispatch10
  libnvinfer-headers-dev libnvinfer-headers-plugin-dev
  libnvinfer-headers-python-plugin-dev libnvinfer-lean-dev libnvinfer-lean10
  libnvinfer-plugin-dev libnvinfer-plugin10 libnvinfer-safe-headers-dev
  libnvinfer-vc-plugin-dev libnvinfer-vc-plugin10
  libnvinfer-win-builder-resource10 libnvinfer10 libnvonnxparsers-dev
  libnvonnxparsers10 python3-libnvinfer python3-libnvinfer-dev
  python3-libnvinfer-dispatch python3-libnvinfer-lean
The following NEW packages will be installed:
  libnvinfer-bin libnvinfer-dev libnvinfer-dispatch-dev libnvinfer-dispatch10
  libnvinfer-headers-dev libnvinfer-headers-plugin-dev
  libnvinfer-headers-python-plugin-dev libnvinfer-lean-dev libnvinfer-lean10
  libnvinfer-plugin-dev libnvinfer-plugin10 libnvinfer-safe-headers-dev
  l

Let's run it again.


In [8]:
import numpy as np
import onnxruntime as ort


sess = ort.InferenceSession("model.onnx",
                            providers=["TensorrtExecutionProvider"])
dummy = {"INPUT__0": np.random.rand(1, 3, 224, 224).astype(np.float32)}
out = sess.run(None, dummy)

print(f"providers={sess.get_providers()}  output={out[0].shape}")

providers=['TensorrtExecutionProvider', 'CPUExecutionProvider']  output=(1, 1000)
